In [ ]:
import json
import pathlib

import geopandas as gpd
import pandas as pd
import pydeck as pdk

In [ ]:
DATA_ROOT_PATH = pathlib.Path().absolute().parent / "data"

In [ ]:
community_next_globus_info = gpd.read_parquet(
    DATA_ROOT_PATH / "processed" / "german_communities.parquet",
)
community_next_globus_info["distance_km_fmt"] = community_next_globus_info["distance"].apply(
    lambda d: f"{(d / 1000):.2f}",
)
community_next_globus_info["distance"] = (
    community_next_globus_info["distance"].round().astype("int32")
)
community_next_globus_info

In [ ]:
globus_positions = pd.read_parquet(DATA_ROOT_PATH / "processed" / "globus_positions.parquet")
globus_positions["geometry"] = list(
    zip(globus_positions["lon"], globus_positions["lat"], strict=False),
)
globus_positions["lon_fmt"] = globus_positions["lon"].apply(lambda d: f"{d:.6f}")
globus_positions["lat_fmt"] = globus_positions["lat"].apply(lambda d: f"{d:.6f}")
globus_positions

In [ ]:
extended_community_info = community_next_globus_info.join(
    globus_positions[["name", "location"]],
    on="globus_row",
).drop(columns=["globus_row"])
extended_community_info

In [ ]:
INITIAL_VIEW_STATE = pdk.ViewState(latitude=51.3, longitude=10, zoom=6, max_zoom=16, bearing=0)

geojson = pdk.Layer(
    "GeoJsonLayer",
    extended_community_info,
    id="communities",
    opacity=0.5,
    stroked=True,
    filled=True,
    get_fill_color=[
        "(distance / 200000) * 255",
        "255 - ((distance / 200000) * 255)",
        "(distance / 200000) * 255",
    ],
    get_line_color=[0, 0, 0],
    auto_highlight=True,
    pickable=True,
)
point_layer = pdk.Layer(
    "ScatterplotLayer",
    globus_positions,
    id="globus",
    opacity=0.8,
    stroked=True,
    filled=True,
    get_fill_color=[255, 255, 255],
    get_position="geometry",
    radius_scale=1,
    radius_min_pixels=5,
    radius_max_pixels=10,
    get_radius=10,
    line_width_min_pixels=1,
    auto_highlight=True,
    pickable=True,
)

pdk_deck = pdk.Deck(
    layers=[geojson, point_layer],
    initial_view_state=INITIAL_VIEW_STATE,
    map_provider=None,
)

In [ ]:
def _round_floats(obj: object, decimals: int = 6) -> object:
    """Recursively round float leaves to ``decimals`` places to shrink the serialized JSON."""
    if isinstance(obj, float):
        return round(obj, decimals)
    if isinstance(obj, list):
        return [_round_floats(v, decimals) for v in obj]
    if isinstance(obj, dict):
        return {k: _round_floats(v, decimals) for k, v in obj.items()}
    return obj


deck_dict = _round_floats(json.loads(pdk_deck.to_json()))
(DATA_ROOT_PATH / "processed" / "deck.json").write_text(
    json.dumps(deck_dict, separators=(",", ":")),
)